# Module 6: Managed Deep Agents

> Part of the **Modular Workshops** series. Standalone, ~25 min.

In Module 3 you shipped the deep agent yourself: you wrote the harness in
`agents/deep_agent/agent.py` (backend, store, memory routes, skills, subagent),
registered it in `langgraph.json`, and ran `langgraph deploy` to host it on a
server you operate.

**Managed Deep Agents (MDA)** is the hosted alternative. You author the agent as
a *directory*, test it with `mda dev`, and ship it with `mda deploy`. The managed
runtime provides the backend, store, checkpointer, memory, skills, and system
prompt — the pieces you wired by hand in Module 3.

This module ports that *same* research agent to MDA. Same tools, same governed
OpenAI model, same skills and subagent — different hosting. The finished project
lives in `agents/managed_deep_agent/`; this notebook walks you through it and the
commands you run in a terminal.

> MDA is in **public beta** and runs on **US LangSmith Cloud only**.

In [ ]:
# Run the shell commands in this notebook from the workshop root (where you'd run `mda`).
import os
from pathlib import Path

root = Path.cwd()
if root.name == "modules":
    os.chdir(root.parent)
print("cwd:", Path.cwd())

---
# Part 1. Why managed?

Both paths give you a hosted agent behind the LangGraph/Agent Server API. The
difference is who operates the runtime.

| | **Self-hosted** (Module 3) | **Managed** (this module) |
|---|---|---|
| Define with | `create_deep_agent(...)` | `define_deep_agent(...)` |
| Backend / store / checkpointer | you wire them (`CompositeBackend`, `StoreBackend`) | runtime injects them |
| Memory / skills / system prompt | constructor args + files | declared by **file location** |
| Config file | `langgraph.json` (you write it) | generated for you |
| Ship with | `langgraph deploy` | `mda deploy` |
| Region / isolation | your choice | US LangSmith Cloud only |

**When to stay self-hosted (`langgraph deploy`):** you need custom server routes,
your own auth wiring inside the deployment, stronger isolation, maximum
scalability, or a non-US region.

**When to use MDA:** you want a code-first Deep Agent on LangSmith without
operating a server, and the managed defaults fit.

---
# Part 2. Install the CLI

The `mda` CLI compiles the project directory into a managed LangGraph app and
deploys it. Install it once (both packages ship the same `mda` binary):

```bash
# Python projects
uv tool install --prerelease allow managed-deepagents

# or TypeScript projects
npm install -g managed-deepagents@dev
```

Check it's on your PATH:

In [ ]:
!mda --version

---
# Part 3. An agent is a directory

With MDA, **a file's location determines its role**. Here is the ported project:

```text
agents/managed_deep_agent/
  agent.py          define_deep_agent(...) — the entry; exports `agent`
  instructions.md   always-on system prompt (Context Hub)
  tools/search.py   the tavily_search tool
  skills/           linkedin-post/ + twitter-post/ (synced to Context Hub)
  memory.py         define_memory(scope="agent") — durable /memories/agent/ tree
  identity.py       who may call the deployment
  pyproject.toml    this project's own dependencies
  .env              LangSmith + tool keys (gitignored; see .env.example)
```

Compare it to the Module 3 agent. Each `create_deep_agent` argument becomes a
file or an author-set field:

| Module 3 (`create_deep_agent`) | Module 6 (`define_deep_agent`) |
|---|---|
| `system_prompt=` + `memory=["./AGENTS.md"]` | `instructions.md` |
| `tools=[tavily_search]` | `tools/search.py`, imported in `agent.py` |
| `skills=["./skills/"]` | `skills/<name>/SKILL.md` |
| `backend=CompositeBackend(routes={"/memories/": StoreBackend()})` | `memory.py` |
| `subagents=[{research-agent}]` | `subagents=[...]` (unchanged) |
| `model` from `utils.models` | gateway client inline in `agent.py` |

> We dropped the Module 3 `interrupt_on` (HITL) to keep this port focused. To add
> it back, pass `interrupt_on={"write_file": True, "edit_file": True}` to
> `define_deep_agent` — the managed runtime owns the checkpointer, so no extra
> setup is needed.

Open `agents/managed_deep_agent/agent.py` and read it — notice what is **absent**:
no backend, store, checkpointer, memory, or skills wiring. Those are managed
fields the runtime injects; setting them here is an error.

In [ ]:
!cat agents/managed_deep_agent/agent.py

The tool is a plain LangChain `@tool`, imported into the agent entry exactly
as in a normal local project. Because `mda deploy` archives *this directory* as
the build root, the project is self-contained — it carries its own search helper
rather than importing the workshop's `utils/`.

In [ ]:
!cat agents/managed_deep_agent/tools/search.py

Memory and identity are declared by their own files. `memory.py` opts the
agent into a durable, deployment-shared tree at `/memories/agent/`; `identity.py`
says a LangSmith workspace API key authenticates callers.

In [ ]:
!cat agents/managed_deep_agent/memory.py
print("\n----- identity.py -----\n")
!cat agents/managed_deep_agent/identity.py

---
# Part 4. Using another provider

`mda init` scaffolds agents on `anthropic:claude-sonnet-4-6` by default. This
workshop runs on OpenAI, routed through the **LangSmith Gateway** so the same
workspace policies from Module 3 §1 (PII / secrets redaction) govern it.

You configure that by passing a **model client** to `define_deep_agent` instead
of a model string — the same `init_chat_model(..., base_url=...)` block you used
in `utils/models.py`:

```python
# agents/managed_deep_agent/agent.py
model = init_chat_model(
    model="gpt-5.6-terra",
    model_provider="openai",
    base_url="https://gateway.smith.langchain.com/openai",
    use_responses_api=True,
    api_key=os.environ["LANGSMITH_API_KEY_GATEWAY"],
)

agent = define_deep_agent(name="managed-research-assistant", model=model, ...)
```

Two things to know:

- **`LANGSMITH_API_KEY_GATEWAY`** holds your LangSmith key under a non-reserved
  name. MDA (like `langgraph deploy`) strips the reserved `LANGSMITH_API_KEY`
  from the deployment, so the gateway client reads the key from the aliased name.
  Put both in `.env` (see `.env.example`).
- There was once an `mda init --gateway` scaffold flag; it was removed. The
  supported way to route through the gateway is to configure the model client
  explicitly, as above.

To use a provider directly instead (no gateway governance), you would pass a
string like `model="anthropic:claude-sonnet-4-6"` or `model="openai:gpt-5.6-terra"`
and supply that provider's key in `.env`.

---
# Part 5. Run it locally

`mda dev` compiles the directory, opens LangSmith Studio, and hot-reloads on
edits. Run it from the workshop root:

```bash
mda dev agents/managed_deep_agent
```

In Studio, send the agent a research question and confirm it calls
`tavily_search` and delegates to the `research-agent` subagent — the same
behavior as the Module 3 agent.

> `mda dev` needs `uv` on your PATH for Python projects; it resolves the local
> LangGraph dev server itself (no global `langgraph` needed). Restart it after
> adding a new managed file (`memory.py`, `tools/mcp.py`, …) — those are picked
> up at compile time, not by hot reload.

You can also compile without running, to check the project is valid:

```bash
mda build agents/managed_deep_agent
```

---
# Part 6. Deploy to LangSmith (optional)

When you're ready, deploy the directory. `mda deploy` compiles the project, syncs
`instructions.md` and `skills/` to Context Hub, uploads the build, waits for
`DEPLOYED`, and reconciles schedules.

```bash
mda deploy agents/managed_deep_agent
```

Useful flags:

- `--name <name>` — deployment name (defaults to the agent's `name=`)
- `--deployment-type prod` — production deployment (default is `dev`)
- `--workspace-id <id>` — when your key requires workspace selection

`mda deploy` prints both the Agent Server URL (to call the agent) and the
LangSmith dashboard URL (to inspect builds, revisions, and traces). The deployed
agent is governed by the Module 3 gateway policy from its first request, because
its model client routes through the gateway `base_url`.

> Requires a `LANGSMITH_API_KEY` with deployment permissions (a service key,
> `lsv2_sk_...`) in the project `.env`.

To tear it down later (**destructive** — removes the deployment, its tracing
project, Context Hub repo, and memory):

```bash
mda delete agents/managed_deep_agent
```

## Recap

| | Self-hosted (Module 3) | Managed (Module 6) |
|---|---|---|
| **Define** | `create_deep_agent(...)` in `agent.py` | `define_deep_agent(...)` in `agent.py` |
| **Harness** | you wire backend / store / memory routes | runtime injects them |
| **System prompt** | `system_prompt=` + `AGENTS.md` | `instructions.md` |
| **Skills** | `skills=["./skills/"]` | `skills/<name>/SKILL.md` → Context Hub |
| **Memory** | `StoreBackend()` route | `memory.py` → `/memories/agent/` |
| **Provider** | gateway client in `utils/models.py` | gateway client inline in `agent.py` |
| **Config** | `langgraph.json` (you write it) | generated by `mda` |
| **Ship** | `langgraph deploy` | `mda deploy` |

**The managed pattern:** an agent is a directory, a file's location is its role,
and `mda deploy` turns that directory into a hosted, governed agent — no harness
wiring and no `langgraph.json` to maintain.

For the full surface (sandboxes, Slack channels, schedules, MCP servers,
connections, Supabase identity, Harbor evals), see the `managed-deep-agents`
skill and the [MDA quickstart](https://docs.langchain.com/langsmith/managed-deep-agents-quickstart).